# Lesson 13 - Agent Memorí


## Setup

Dis notebook go show how to build travel booking agent wey get **persistent memory** using **Microsoft Agent Framework** (MAF).

You go learn how different kain agent memory — working, short-term, and long-term — dey affect how agent dey keep and use information for different talks.

**Prerequisites:**
- Microsoft Foundry project wey get deployed chat model (like `gpt-4.1-mini`).
- Logged in wit Azure CLI — run `az login` for your terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Kain Kain Memory We Agent Fit Get

AI agents fit use different kinds memory, each one get im own purpose:

### Working Memory
Na di conversation thread wey dey — di messages wey dem exchange for one session. Di agent fit check dia messages before for dat same thread to make sure say everything dey okay. For MAF, na wetin **`agent.create_session()`** create, e go give one `AgentSession`.

### Short-Term Memory
Na information wey go last only for di time wey task or session dey run but e no go stay forever. For example, the agent fit collect facts as e dey do multi-turn planning talk and use am take create final itinerary.

### Long-Term Memory
Preferences and facts wey go stay **across sessions**. If person come back again, e no suppose dey tell dem dietary restrictions or travel style again. Long-term memory usually dey backed by one external store — like database, file, or vector index — and e dey come to agent through tools.


## Working Memory wit Sessions

Di simplest kind memory na di conversation session. Wen you dey pass di same session (wey you create thru `agent.create_session()`) go every `agent.run()` calls wey follow, di agent go fit see di full history of dat conversation and e go fit remember wetin dem talk before.

Make we create travel agent and show how working memory dey work.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Di agent recall di budget well becos both messages dey share di same session. Dis na **working memory** — e dey only exist for di lifetime of di session.

### Wetin go happen if na new thread?

If we create **new** session, di agent no go get memory of di previous conversation:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Long-Term Memory Pattern

To sabi wetin user like **across sessions**, we need wan durable place wey no dey inside conversation thread. Di agent go fit use **tools** — functions wey e fit call to save and find information.

Down here, we go build simple in-memory preference store (for production, you fit use database or vector index) then make am tools wey di agent fit use.

### Architecture
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — First-time user books an anniversary trip

Sarah show for di first taim. Di agent suppose save her tins wey she like through di tools dem and use am recommend beta hotels.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenario 2 — Sarah return weeks afta

Sarah start one **brand-new thread** (wey dey simulate new session). Di working memory empty, but di long-term preference store still get her info. Di agent suppose find am come use am take personalize recommendations.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Summary

For dis lesson, you don learn about three kain agent memory dem and how mek you fit implement dem with Microsoft Agent Framework:

| Memory Type | MAF Mechanism | Lifetime |
|---|---|---|
| **Working** | `agent.create_session()` | One conversation |
| **Short-term** | Accumulate context inside one thread | One task / session |
| **Long-term** | External store wey you fit use `@tool` functions access | Across sessions |

### Key Takeaways
1. **`agent.create_session()`** dey give working memory — agent fit see all di history of conversation inside one session.
2. **New sessions dey lose context** — if no long-term memory, agent no fit remember past conversations.
3. **`@tool` functions** dey join di gap — dem go allow agent save and collect information from one persistent store.
4. **Personalization dey improve as time dey go** — di more preferences wey store, di beta recommendations wey agent fit give.

### Real-World Applications
- **Customer Service**: Remember customer history and wetin dem like
- **Personal Assistants**: Keep context for days or weeks
- **Healthcare**: Track patient information and wetin dem like
- **E-commerce**: Personalized shopping based on history

### Next Steps
- Change di in-memory dict to database or vector store (like Azure AI Search)
- Add memory expiration for time-sensitive information
- Build multi-agent systems with shared memory
- Check out di Cognee notebook for knowledge-graph-backed memory


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
